# 02 — Data Quality Checks on First-Transaction Data

This notebook checks **which of the data-quality issues found in `01_eda_raw` survive after restricting to each customer's first transaction**. It does *not* clean the data — it only quantifies what remains, so we can decide how to handle each issue before building customer-level features.

Source: `data/processed/first_transaction_churn.csv`, produced by `data/processing/prepare_churn_data.py` (one row per line item of each customer's **first** invoice, plus a `churn` label). The raw figures referenced below come from `01_eda_raw`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

PROCESSED = Path('..') / 'data' / 'processed'
ft = pd.read_csv(PROCESSED / 'first_transaction_churn.csv', parse_dates=['invoice_date'])

n_rows, n_inv, n_cust = len(ft), ft['invoice'].nunique(), ft['customer_id'].nunique()
print(f'Rows: {n_rows:,} | first-transaction invoices: {n_inv:,} | customers: {n_cust:,}')
ft.head()

## 1. Missing values (`customer_id`, `description`)

Raw data had 22.77% missing `customer_id` and 0.41% missing `description`. Rows with no `customer_id` are dropped when building the first-transaction table, so this should be 0; we also check whether any first-invoice line still lacks a description.

In [ ]:
print('Missing values in first_txn  (raw 01 figure in brackets):')
print(f"  customer_id : {ft['customer_id'].isna().sum():>6,} ({ft['customer_id'].isna().mean()*100:5.2f}%)   [raw: 22.77%]")
print(f"  description : {ft['description'].isna().sum():>6,} ({ft['description'].isna().mean()*100:5.2f}%)   [raw:  0.41%]")

## 2. Zero prices (`price == 0`)

Raw data had 6,202 rows with `price == 0` (likely freebies / adjustments).

In [ ]:
zero_price = ft['price'] == 0
print(f"price == 0 rows : {zero_price.sum():,} ({zero_price.mean()*100:.2f}%)  across {ft.loc[zero_price,'invoice'].nunique():,} invoices   [raw: 6,202 rows]")
ft[zero_price].head(10)

## 3. One stock code → many descriptions

Raw data reported 53.38% of stock codes carrying more than one description — but that count treated *missing* descriptions as a distinct value, so it was inflated. `first_txn` has no missing descriptions (see §1), so this is a cleaner measure of genuinely conflicting labels.

In [ ]:
desc_norm = ft['description'].astype(str).str.strip().str.upper()

code_desc = (ft.assign(description_norm=desc_norm)
               .groupby('stock_code')['description_norm']
               .nunique()
               .sort_values(ascending=False)
               .reset_index(name='unique_descriptions'))
n_codes = len(code_desc); n_multi = (code_desc['unique_descriptions'] > 1).sum()
print(f'Stock codes total           : {n_codes:,}')
print(f'Stock codes with >1 desc    : {n_multi:,} ({n_multi/n_codes*100:.2f}%)   [raw: 53.38%, but raw counted missing description as a value]')
code_desc[code_desc['unique_descriptions'] > 1].head(20)

In [ ]:
# Check a couple of examples with the stock_ids that have multiple descriptions
for i in ('20685', '22785', '22845', '84997C'):
    print(f'Number of unique descriptions for {i}: \n {ft[ft['stock_code'] == i]['description'].unique()}')

## 4. One description → many stock codes

Raw data: 5.20% of descriptions mapped to more than one stock code (generic labels like `CHECK`, `DAMAGED`, `?`).

In [ ]:
desc_code = (ft.assign(description_norm=desc_norm)
               .groupby('description_norm')['stock_code']
               .nunique()
               .sort_values(ascending=False)
               .reset_index(name='unique_stock_codes'))
n_desc = len(desc_code); n_multi = (desc_code['unique_stock_codes'] > 1).sum()

print(f'Distinct descriptions       : {n_desc:,}')
print(f'Descriptions with >1 code   : {n_multi:,} ({n_multi/n_desc*100:.2f}%)   [raw: 5.20%]')
desc_code[desc_code['unique_stock_codes'] > 1].head(32)

## 5. Stock code format

Issue #5 from `01`: stock codes are stored in different formats — numbers only, letters only (`POST`, `M`, `DOT`, …), number + letters (product variants such as `84031A`), or other (`gift_0001_80`, `BANK CHARGES`). The **letters-only** and **other** formats are not valid product codes and need processing; number + letters are mostly legitimate variants. Here we check whether the *invalid* formats still appear after keeping only the first transaction.

In [ ]:
# Format composition of the unique stock codes.
u = pd.Series(ft['stock_code'].astype(str).unique())
is_num   = u.str.fullmatch(r'[0-9]+')
is_alpha = u.str.fullmatch(r'[A-Za-z]+')
is_mix   = u.str.fullmatch(r'(?=.*[A-Za-z])(?=.*[0-9])[A-Za-z0-9]+')
is_other = ~(is_num | is_alpha | is_mix)

n = len(u)
print(f'Unique stock codes        : {n:,}')
print(f'  numeric only            : {is_num.sum():>5,} ({is_num.mean()*100:5.2f}%)')
print(f'  letters only            : {is_alpha.sum():>5,} ({is_alpha.mean()*100:5.2f}%)')
print(f'  number + letters (mixed): {is_mix.sum():>5,} ({is_mix.mean()*100:5.2f}%)')
print(f'  other (symbols/spaces)  : {is_other.sum():>5,} ({is_other.mean()*100:5.2f}%)')

# Letters-only and "other" are NOT valid product codes -> these still need processing.
invalid_codes = set(u[is_alpha | is_other])
invalid_row = ft['stock_code'].astype(str).isin(invalid_codes)
print('\nInvalid-format codes (letters-only or other) still present after the 1st-transaction filter:')
print(f'  unique codes : {sorted(invalid_codes)}')
print(f'  line rows    : {invalid_row.sum():,} ({invalid_row.mean()*100:.2f}%)')
print(f'  invoices     : {ft.loc[invalid_row, "invoice"].nunique():,} ({ft.loc[invalid_row, "invoice"].nunique()/n_inv*100:.2f}%)')

## 6. Cancellations (invoices starting with `C`)

These are **kept by design** in the first-transaction table (a customer whose first invoice is a cancellation is still a customer). We just measure how many there are.

In [ ]:
is_cancel = ft['invoice'].astype(str).str.startswith('C')
n_cancel_inv = ft.loc[is_cancel, 'invoice'].nunique()
print(f'Cancellation (C) line rows : {is_cancel.sum():,}')
print(f'Cancellation invoices      : {n_cancel_inv:,} ({n_cancel_inv/n_inv*100:.2f}%)   [kept by design]')

## 7. Outliers in the continuous variables

Tukey 1.5×IQR rule on the positive first-transaction rows (`quantity > 0` & `price > 0`), same definition as `01`'s Section 6.

In [ ]:
clean = ft[(ft['quantity'] > 0) & (ft['price'] > 0)].copy()
print('Tukey IQR outliers (positive first-transaction rows):')
for col in ['quantity', 'price', 'line_total']:
    s = clean[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    is_out = (s < lo) | (s > hi)
    print(f'  {col:11s}: Q1={q1:,.2f}  Q3={q3:,.2f}  upper fence={hi:,.2f}  ->  '
          f'{is_out.sum():,} outliers ({is_out.mean()*100:.2f}%)')

## 8. Test products (`TEST*` stock codes)

Placeholder records left in the retailer's database — the description is the literal string `This is a test product.`. These are not real purchases.

The §5 format check does **not** catch them: `TEST001` is letters+digits, so it falls in the "number + letters (mixed)" bucket alongside genuine product variants such as `84031A`. Here we count how many reach the first-transaction table, and — because a test row that is only *part* of a real invoice matters far less than one that *is* the whole invoice — how many customers have nothing but test rows in their first transaction.

In [ ]:
is_test = ft['stock_code'].astype(str).str.upper().str.startswith('TEST')

print(f'Test rows in first_txn      : {is_test.sum():,} ({is_test.mean()*100:.4f}%)  '
      f'codes {sorted(ft.loc[is_test, "stock_code"].unique())}')
print(f'Customers with a test row   : {ft.loc[is_test, "customer_id"].nunique():,}')

# Customers whose first transaction consists ONLY of test rows -> entirely fake customers.
all_test = ft.groupby('customer_id')['stock_code'].apply(
    lambda s: s.astype(str).str.upper().str.startswith('TEST').all())
print(f'Customers whose whole 1st txn is test rows : {all_test.sum():,}  {list(all_test[all_test].index)}')

ft[is_test]

## Summary — what survives the first-transaction filter

| # | Issue | Raw (01) | First transactions | Status |
|---|---|---|---|---|
| 1 | Missing `customer_id` / `description` | 22.77% / 0.41% | 0 / 0 | ✅ resolved |
| 2 | `price = 0` | 6,202 rows | 7 rows (7 invoices) | ⚠️ nearly gone |
| 3 | stock_code → >1 description | 53.38%\* | 9.10% | ⚠️ reduced |
| 4 | description → >1 stock_code | 5.20% | 0.70% | ⚠️ reduced |
| 5 | stock_code format (letters-only / other invalid) | present | 7 invalid codes, 480 rows (0.38%), 452 invoices | ❌ still present, needs processing |
| 6 | cancellations (`C` invoices) | 8,292 invoices | 274 invoices (5.13%) | ❌ kept by design |
| 7 | outliers (IQR) | present | ~4–8% per column | ❌ still present |
| 8 | test products (`TEST*`) | not measured in `01` | 4 rows, 4 customers — each customer's *entire* first transaction | ❌ still present, missed by §5 |

\* the raw figure counted missing descriptions as a distinct value, so it was inflated.

**Still to decide handling for, before building customer-level features:** zero prices (2), conflicting descriptions (3 / 4), invalid stock-code formats (5), cancellations (6), outliers (7), and test products (8). Those decisions feed `data/processing/build_features.py` (next step).

## Resolution plan — how each issue will be handled

How I intend to resolve each issue before building customer-level features. Numbering matches the checks above.

1. **Missing values (`customer_id`, `description`)** — ✅ Already solved. The first-transaction filter drops every row with a missing `customer_id`, and no missing `description` remains (both 0% in §1).

2. **Zero prices (`price == 0`)** — Remove the 7 rows where `price == 0`.

3. **One stock code → many descriptions** — Replace each `stock_code`'s descriptions with its **most common description**, applied to all of that code's rows. Most of the differences are commas, synonyms, or typos, so the modal description is the reliable label.

4. **One description → many stock codes** — Treat `stock_code` as the absolute source of truth for product identity; leave descriptions as they are.

5. **Invalid stock-code formats** — The letters-only / non-product codes are `[POST, M, D, ADJUST, PADS, DOT, BANK CHARGES]` (480 rows). Remove those instances.

6. **Cancellations (`C` invoices)** — **Remove them.** Here a customer's *first* transaction is itself a cancellation, but this table contains **cancellations only** (there is no matching earlier purchase in our data). The assumption is that a cancellation was almost certainly preceded by an actual purchase — but since that prior transaction is not present, we would be treating a cancellation as the customer's "very first transaction," which is wrong. We therefore remove these customers rather than mislabel them.

7. **Outliers** — **Keep them.** We are using a **tree-based model**, which splits on feature thresholds and is inherently robust to extreme values — the model handles the outliers itself. So we leave them untouched rather than clip, winsorise, or drop them.

8. **Test products (`TEST*`)** — **Remove them.** These are placeholder records, not purchases, so they carry no information about real customer behaviour. Each of the 4 rows is a customer's *entire* first transaction (§8), so removing the rows removes those 4 customers — which is the right outcome: they are not real customers to predict churn for. Matching is on the `TEST` prefix rather than the literal `TEST001`, because the raw file also contains `TEST002`; it does not reach this table today, but a prefix rule catches it if that ever changes. Note this is **not** covered by the issue-5 format rule: `TEST001` is letters+digits, which §5 classifies as a valid product variant.

In [ ]:
clean.groupby(['customer_id'])['stock_code'].nunique().sort_values(ascending=False)